# 03. CLIP window sweep на 1,000 queries

Этот notebook читает сохраненный финальный CLIP sweep из `results/charades_sta_sweep_1000/summary.csv`. Он не запускает CLIP inference. Цель - сравнить temporal window size, stride и aggregation на одной фиксированной 1,000-query Charades-STA subset.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

sweep = pd.read_csv(ROOT / 'results/charades_sta_sweep_1000/summary.csv')
cols = ['config_name', 'window_size', 'stride', 'aggregation', 'evaluated_queries', 'unique_videos', 'R@1_IoU_0.3', 'R@1_IoU_0.5', 'R@1_IoU_0.7', 'mIoU', 'inference_time_sec']
sweep[cols]


,config_name,window_size,stride,aggregation,evaluated_queries,unique_videos,R@1_IoU_0.3,R@1_IoU_0.5,R@1_IoU_0.7,mIoU,inference_time_sec
0,clip_w8_s4_mean,8.0,4.0,mean,1000,363,0.509,0.393,0.181,0.347777,309.686636
1,clip_w16_s8_mean,16.0,8.0,mean,1000,363,0.625,0.260,0.096,0.349690,22.592478
2,clip_w32_s16_mean,32.0,16.0,mean,1000,363,0.395,0.014,0.000,0.281429,21.506191
3,clip_w16_s8_max,16.0,8.0,max,1000,363,0.602,0.232,0.077,0.338619,20.922287


In [2]:
metric_cols = ['R@1_IoU_0.3', 'R@1_IoU_0.5', 'R@1_IoU_0.7', 'mIoU']
best = []
for metric in metric_cols:
    row = sweep.loc[sweep[metric].idxmax()]
    best.append({'metric': metric, 'best_config': row['config_name'], 'value': row[metric]})
pd.DataFrame(best)


,metric,best_config,value
0,R@1_IoU_0.3,clip_w16_s8_mean,0.62500
1,R@1_IoU_0.5,clip_w8_s4_mean,0.39300
2,R@1_IoU_0.7,clip_w8_s4_mean,0.18100
3,mIoU,clip_w16_s8_mean,0.34969


Вывод: `clip_w16_s8_mean` дает лучший coarse retrieval при IoU 0.3, а `clip_w8_s4_mean` дает лучшую stricter localization при IoU 0.5 и 0.7. Эта таблица является основным финальным CLIP result курсовой.


Reproducibility note: heavy experiment runner - `scripts/run_clip_sweep.py`, но этот notebook намеренно читает сохраненный public result file и не rerun-ит inference.
